In [1]:
import pandas as pd
import os
from mistralai import Mistral
import httpx
import re
from mistralai.models import OCRResponse
from IPython.display import Markdown, display
from openai import OpenAI
import numpy as np
import asyncio
from portkey_ai import Portkey , AsyncPortkey

In [2]:
def openai_client(api_key):
    client = Portkey(base_url=os.environ.get("PORTKEY_BASE_URL"), api_key=os.environ.get("PORTKEY_API_KEY"))
    return client

def openai_message(text, base64_encoding):
     messages=[
        {
            "role": "user",
            "content": [
                { "type": "text", "text": text },
                {
                    "type": "image_url",
                    "image_url": {
                        "url": base64_encoding,
                    },
                },
            ],
        }
    ]
     return messages


def base_64_list_per_page(ocr_response,page_no):   
    """
    Gets base64 encodings of images per page, all the images are in the same order as they are in the page
    """
    base64_strings=[]
    base64_for_gpt=[]
    for image in ocr_response.pages[page_no-1].images:
        base64_for_gpt.append(image.image_base64)
        base64_img=image.image_base64.split(',')[1]
        base64_strings.append(base64_img)
    return base64_strings, base64_for_gpt

def base_64_list_all_pages(ocr_response):
    """
    Gets base64 encodings of all images present in the PDF in form of list and all the images are in the order.
    """
    base_64_all=[]
    total_no_of_pages=len(ocr_response.pages)
    for page in range(1, total_no_of_pages+1):
        a,b=base_64_list_per_page(ocr_response=ocr_response,page_no=page)
        base_64_all.extend(b)
    return base_64_all

async def base64_to_image_content(prompt, base64_encoding, api_key):
    client = AsyncPortkey(base_url=os.environ.get("PORTKEY_BASE_URL"), api_key=os.environ.get("PORTKEY_API_KEY"))
    response = await client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": prompt},
                    {"type": "image_url", "image_url": {
                        "url": base64_encoding }}
                ]
            }
        ],
        max_tokens=500
    )
    return response.choices[0].message.content.strip()

async def process_all_images(prompt, api_key, base64_img_list):
    """
    Asynchronously processes all Base64 images using OpenAI API and returns the generated contents.
    """
    tasks = [
        base64_to_image_content(prompt=prompt, base64_encoding=img, api_key=api_key)
        for img in base64_img_list
    ]
    return await asyncio.gather(*tasks)

def replace_markdown_images_with_content(input_text, generated_contents):
    
    if not isinstance(input_text, str):
        raise ValueError("Expected 'input_text' to be a string, got None or invalid type.")

    image_pattern = re.compile(r"!\[(.*?)\]\((.*?)\)")
    content_iter = iter(generated_contents)
    index = 1

    def replacer(match):
        nonlocal index
        try:
            content = next(content_iter)
            result = f"\nIMAGE {index} begins...\n{content}\nIMAGE {index} ends...\n"
            index += 1
            return result
        except StopIteration:
            return match.group(0)

    return image_pattern.sub(replacer, input_text)


In [3]:
def mistral_client(api_key):
       client = Mistral(api_key=api_key, 
    client=httpx.Client(verify=False)
)
       return client

def mistral_signed_pdf_url(pdf_path:str, api_key):
    client=mistral_client(api_key=api_key)
    uploaded_pdf=client.files.upload(
    file={
        "file_name": pdf_path,
        "content": open(pdf_path, "rb"),
    },
    purpose="ocr"
)
    client.files.retrieve(file_id=uploaded_pdf.id)
    signed_url = client.files.get_signed_url(file_id=uploaded_pdf.id)

    return client,signed_url

def mistral_ocr_pdf(pdf_path:str, api_key):
    client=mistral_client(api_key=api_key)
    if pdf_path.startswith('https://'):
        ocr_response = client.ocr.process( model="mistral-ocr-latest",
        document={
            "type": "document_url",
            "document_url": pdf_path,
        }, 
        include_image_base64=True
    )
        
    else:
        client,signed_url=mistral_signed_pdf_url(pdf_path=pdf_path, api_key=api_key)
        ocr_response = client.ocr.process(
        model="mistral-ocr-latest",
        document={
            "type": "document_url",
            "document_url": signed_url.url,
        }, 
        include_image_base64=True
    )
    return ocr_response

async def amistral_ocr_pdf(pdf_path: str, api_key: str):
    client, signed_url = mistral_signed_pdf_url(pdf_path, api_key)

    ocr_response = await client.ocr.process_async(
        model="mistral-ocr-latest",
        document={
            "type": "document_url",
            "document_url": signed_url.url,
        },
        include_image_base64=True
    )

    return ocr_response
    
def aggregate_markdowns(ocr_response):  
    markdowns=""
    for docu in ocr_response.pages:
        markdowns+=f"\n<start_of_page_{str(docu.index+1)}>\n"
        markdowns+=docu.markdown
        markdowns+=f"\n<end_of_page_{str(docu.index+1)}>\n"
    return markdowns

def save_markdown_to_file(markdown_content, output_file):
    """
    Saves the markdown content to a specified file.
    """
    with open(output_file, 'w', encoding='utf-8') as f:
        f.write(markdown_content)
        
    print(f"✅ Markdown content saved to {output_file}")

In [4]:
async def pdf_to_markdown(pdf_path, mistral_api_key, replace_images=True, prompt="Extract all the text from the image"):
    """
    Converts a PDF file to markdown using Mistral OCR.
    
    Parameters:
        pdf_path (str): The path to the PDF file or a URL.
        mistral_api_key (str): The Mistral API key.
        
    Returns:
        str: The markdown content extracted from the PDF.
    """

    ocr_response = mistral_ocr_pdf(pdf_path=pdf_path, api_key=mistral_api_key)
    markdown_text = aggregate_markdowns(ocr_response)
    print("✅ Markdown text extracted from the PDF")
    
    if replace_images:
        generated_contents = await process_all_images(
        prompt=prompt,
        api_key=os.environ.get("OPENAI_API_KEY"),
        base64_img_list=base_64_list_all_pages(ocr_response)
    )
        print("✅ Image content extracted and replaced in markdown")
        
        markdown_text=replace_markdown_images_with_content(input_text=markdown_text,generated_contents=generated_contents)

    os.makedirs("pdf_markdowns", exist_ok=True)
    file_name=os.path.basename(pdf_path)[:-4]
    save_markdown_to_file(markdown_content=markdown_text, output_file=f"pdf_markdowns/{file_name}.md")
    return markdown_text


async def process_multiple_pdfs(pdf_paths: list, api_key: str, replace_images: bool = True, prompt="Extract all the text from the image"):
    tasks = {
        asyncio.create_task(amistral_ocr_pdf(path, api_key)): path
        for path in pdf_paths
    }

    results = {}  # Dictionary to hold PDF path → result
    pending_pdfs = set(pdf_paths)

    for completed_task in list(tasks.keys()):
        try:
            result = await completed_task
            pdf_path = tasks[completed_task]
            print(f"✅ Finished processing: {pdf_path}")
            pdf_content= aggregate_markdowns(result)
            if replace_images:
                generated_contents = await process_all_images(
                    prompt=prompt,
                    api_key=os.environ.get("OPENAI_API_KEY"),
                    base64_img_list=base_64_list_all_pages(result)
                )
                results[pdf_path] = replace_markdown_images_with_content(input_text=pdf_content,generated_contents=generated_contents)
            else:
                results[pdf_path] = pdf_content

            os.makedirs("pdf_markdowns", exist_ok=True)
            file_name=os.path.basename(pdf_path)[:-4]
            save_markdown_to_file(markdown_content=results[pdf_path], output_file=f"pdf_markdowns/{file_name}.md")

        
        except Exception as e:
            pdf_path = tasks.get(completed_task, "<unknown>")
            print(f"❌ Error processing {pdf_path}: {e}")
            results[pdf_path] = None

        pending_pdfs.discard(pdf_path)
        if pending_pdfs:
            print(f"⏳ Still waiting on: {', '.join(pending_pdfs)}")

    return results

### PDF extraction for single PDF

In [7]:
pdf_path=r"..\\..\\Sample PDFs\\Multi-Column.pdf"

# There is a folder named sample_pdfs in the same directory as this script, you can change the path to your own PDF file.
# Or you can also use the PDF saved anywhere in your system
# You can also use a URL of the PDF file, just make sure it is accessible.                     

In [9]:
await pdf_to_markdown(pdf_path=pdf_path, mistral_api_key=os.environ.get("MISTRAL_API_KEY"), replace_images=True)


# Put replace_images=True if you want to replace the images present in the image with the informaton extracted from it.
# The markdowns will be saved in a folder named pdf_markdowns in the same directory as this notebook.

✅ Markdown text extracted from the PDF
✅ Image content extracted and replaced in markdown
✅ Markdown content saved to pdf_markdowns/Multi-Column.md


'\n<start_of_page_1>\nNestlé\n\nGood food, Good life\n\n\nIMAGE 1 begins...\nCreating Shared Value and Sustainability Report 2023  \nAdvancing regenerative food systems at scale\nIMAGE 1 ends...\n\n\nCreating Shared Value and Sustainability Report 2023\n\nAdvancing regenerative food systems at scale\n<end_of_page_1>\n\n<start_of_page_2>\n# Welcome to the Nestlé Creating Shared Value and Sustainability Report 2023.\n\nExpert voices, case studies and data support detail on our approach and performance, covering the topics most material to our business and stakeholders. Read more about independent assurance of this report on page 71.\n\n\nIMAGE 2 begins...\nI\'m unable to see the content of the image and extract text from it. Could you describe the image or provide the text in another way?\nIMAGE 2 ends...\n\n\n## Contents\n\n2 Chairman and CEO message\n3 Our business model and progress against commitments\n4 Governance\n5 Our material topics\n6 Stakeholder engagement\n7 On the road to ne

### PDF extraction for multiple PDFs

In [5]:
pdf_folder="..\\..\\Sample PDFs"    # Mention the folder path where your PDFs are stored or 
                            # you can create one folder in the same directory as this script and put your PDFs there.
                            # or you can use sample_pdfs folder for demo run 

In [6]:
import glob

pdf_paths=glob.glob(r"..\\..\\Sample PDFs\\*.pdf")  # This will get all the PDF files in the sample_pdfs folder

In [ ]:
results=await process_multiple_pdfs(pdf_paths=pdf_paths, api_key=os.environ.get("MISTRAL_API_KEY"), replace_images=True)

# Put replace_images=True if you want to replace the images present in the image with the informaton extracted from it.
# The markdowns will be saved in a folder named pdf_markdowns in the same directory as this notebook.

✅ Finished processing: ..\\..\\Sample PDFs\AMER_A&D_Defense_General Dynamics_Q1 (Jan-Mar).pdf
✅ Markdown content saved to pdf_markdowns/AMER_A&D_Defense_General Dynamics_Q1 (Jan-Mar).md
⏳ Still waiting on: ..\\..\\Sample PDFs\OP-RAG.pdf, ..\\..\\Sample PDFs\Analyst Report.pdf, ..\\..\\Sample PDFs\Financial Statements.pdf, ..\\..\\Sample PDFs\InfoEdge_Annual_Report_2025.pdf, ..\\..\\Sample PDFs\Research Paper.pdf, ..\\..\\Sample PDFs\Invoice.pdf, ..\\..\\Sample PDFs\Form.pdf, ..\\..\\Sample PDFs\PPT.pdf, ..\\..\\Sample PDFs\Scanned.pdf
✅ Finished processing: ..\\..\\Sample PDFs\Analyst Report.pdf
✅ Markdown content saved to pdf_markdowns/Analyst Report.md
⏳ Still waiting on: ..\\..\\Sample PDFs\OP-RAG.pdf, ..\\..\\Sample PDFs\Financial Statements.pdf, ..\\..\\Sample PDFs\InfoEdge_Annual_Report_2025.pdf, ..\\..\\Sample PDFs\Research Paper.pdf, ..\\..\\Sample PDFs\Invoice.pdf, ..\\..\\Sample PDFs\Form.pdf, ..\\..\\Sample PDFs\PPT.pdf, ..\\..\\Sample PDFs\Scanned.pdf
✅ Finished processing: